## Computational Physics -- Problem Set 2
##### _Theodore Landa_

In [ ]:
import numpy as np
import pandas as pd
import math
import time
import matplotlib.pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')

### Problem 1.

_(a)_ A naive implementation of np.sum() gave me $29360128.0$. I re-ordered them by absolute value (descending), re-summed with np.sum(), and got $29999988.0$.

_(b)_ I verified the correctness of sum by inspecting the data. In particular, after some hints in recitation about what the data looked like, I used some masks and a plot to find out precisely what the data were and determined exactly what the sum should be. Naively, though, the higher number should be more correct, given that the computer truncates rather than rounding.

_(c)_ The unsorted manual sum gave $2963903.0$, and the sorted manual sum gave the true answer of $30000000.0$. I also tried using math.fsum() on the unsorted data, which gave me $30000000.0$ as well. Numpy seems to do its sums by pairing numbers and summing them, and then pairing again, etc; I see no other way besides grouping the small magnitude numbers and the big magnitude numbers (located on opposite sides in the sorted array) and summing that would introduce numerical error.

In [ ]:
file = 'ps02.csv'
df = pd.read_csv(file, header = None)

In [ ]:
data = np.squeeze(df.to_numpy())
masklo = data < 1.0
maskhi = data > 1.0
print(data[masklo])
print(data[maskhi])
print(data.size)

In [ ]:
plt.plot(np.linspace(0, data.size - 1, data.size), data, 'k')
plt.xlabel('Index')
plt.ylabel('Value')
plt.title('CSV File Data')

In [ ]:
print(np.sum(data))
print(math.fsum(data))
sorter = np.abs(data)
sortidx = np.argsort(sorter)[::-1]
final = data[sortidx]
print(np.sum(final))

def manual_sum(data):
    sum = 0
    for j in data:
        sum += j
    return sum

print(manual_sum(data))
print(manual_sum(final))

### Problem 2.

My results are indeed consistent with what I got in problem set 1! The result my bisection code gets for $\delta$ is $1.1102\times10^{-16}$, where $\ln(\delta) = -36.7368$. My code is "verbose," and is agnostic about which of $x_1,x_2$ is smaller or larger!

In [ ]:
def f(x):
    return 1. + np.exp(x) > 1

In [ ]:
def find_root(f, x1, x2, tol=1e-6):
    if f(x1):
        xtrue, xfalse = x1, x2
    else:
        xtrue, xfalse = x2, x1
    assert f(xtrue) & ~f(xfalse)
    print('('+str(xfalse)+', '+str(xtrue)+')')
    xmid = (xtrue + xfalse)/2
    if (np.abs(xtrue - xfalse) < tol):
        return xmid
    elif f(xmid):
        return find_root(f, xmid, xfalse)
    else:
        return find_root(f, xmid, xtrue)

In [ ]:
exp = find_root(f, 0, -50)
print(exp)
print(np.exp(exp))
print(find_root(f, -50, 0) == exp)

### Problem 3.
_(a)_ We know that $\frac{df}{dx}=2x-1$, and so the derivative evaluated at $1$ should be $1$. My code returns $1.010000000000001$ for $h=10^{-2}$ because we haven't truly taken the limit; we're just approximating.

_(b)_ Starting from the $-4$th power and going down to the $-14$th, we get derivative values of $[1.0001,     1.000001,   1.,         1.00000008, 1.0000889,  0.99920072]$. We also have a plot, which is way better! The accuracy improves, but then worsens, because we get closer and closer to a linear approximation at 1, until $h$ gets small enough that numerical error starts piling up with the catastrophic subtraction in the numerator.

In [ ]:
def f1(x):
    return x*(x-1)

In [ ]:
def ddx(f, h, x):
    return (f(x+h)-f(x))/h

In [ ]:
print(ddx(f1, 1e-2, 1))

In [ ]:
h = np.logspace(0, -16, 1000)
der = ddx(f1, h, 1)
plt.loglog(h, np.abs(der - 1), 'k')
plt.xlabel(r'Infinitesimal $h$')
plt.ylabel(r'Computed Derivative $\frac{f(1+h)-f(1)}{h}$')
plt.title(r'How Does Accuracy Change with $h$?')

In [ ]:
h1 = 10.**(np.arange(-4, -16, -2))
der1 = ddx(f1, h1, 1)
print(h1)
print(der1)

### Problem 4.
Check out the plots!

In [ ]:
def f2(x):
    return np.sqrt(1-x**2)

In [ ]:
def integ(f, N, start, stop):
    starttime = time.time()
    x = np.linspace(start, stop, N, endpoint=False)
    y = f(x)
    I = np.sum(y)*(stop-start)/N
    endtime = time.time()
    cost = endtime - starttime
    return I, cost

In [ ]:
numNs = 1000
Ns = np.logspace(2, 8, numNs, dtype=np.int32)
Ierrs = np.zeros(numNs)
Itimes = np.zeros(numNs)
for j in range(numNs):
    Ival, Itimes[j] = integ(f2, Ns[j], -1, 1)
    Ierrs[j] = np.abs(Ival-(np.pi/2))

In [ ]:
plt.loglog(Ns, Ierrs, 'k.')
plt.xlabel(r'Number of Evaluations $N$')
plt.ylabel('Computed Integral Numerical Error')
plt.title(r'How Does Accuracy Improve with $N$?')

In [ ]:
plt.loglog(Itimes, Ierrs, 'k.')
plt.xlabel(r'Wall Clock Time $T$')
plt.ylabel('Estimated Integral Numerical Error')
plt.title(r'How Does Accuracy Improve with $T$?')

### Credits.
Nana Miller for helping me figure out CSV files, Erin Shu for pointing out math.fsum, Gabe Rosen-Turits for help with plotting, the matplotlib and numpy documentations for basically everything, and Seth Murchison for emotional support.